<a href="https://colab.research.google.com/github/IrumShehryar/ML-NLP-Coursework/blob/main/nlp/04-neural-network/markov-model-for-text-generation/Project04_Markov_Text_Generation_Large_Corpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [1]:
import numpy as np
import string

# Initialize the distribution dictionaries

In [2]:
pi = {} # Initial distribution representing the start of the sentencee
A1 = {} # First order transition for second word only.
A2 = {} # This will store second order transitions i.e the words after the second word.

# Create the function to fill the above dictionaries

In [5]:
def fill_dict(d, k, v): # Three inputs are dictionary, keys and values. Keys represent some starting words or pair
                        #of words in the second order case as shown in above figure. First we collect the words from text like
                        # happy, sad, content and so on and then assign prob to the words.
  if k not in d:
    d[k] = []
  d[k].append(v)

# Mount the Drive and set the path

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/NLP-course/Course Material/Markov Model for Text Generation

/content/drive/MyDrive/NLP-course/Course Material/Markov Model for Text Generation


# Iterate over the text to fill the dictionaries

In [6]:
# The following lines of codes will populate the dictionaries we initialized above
for line in open('YiLei_Poem.txt'):
  tokens = line.rstrip().lower().split() # get the list of tokenss.

  T = len(tokens)
  for i in range(T):
    t = tokens[i] # Grab the ith token
    if i == 0: # This is the first word in sentence so we would like to update our first word in distribution.
               # we can do it by incrementing the value stored in t by one
      # measure the distribution of the first word by
      pi[t] = pi.get(t, 0.) + 1 # get will ensure that if t is not present in dictionary, it automatically get the count of 0.
                                          # The first step is only to gather the counts. After getting through the whole dataset we can
                                          # normalize this distribution.
    else:
      t_1 = tokens[i-1] # If i is not 0, we will grab the previous word in sentence
      if i == T - 1: # check if we are at the end of the sentence. if i = T-1, we are at the end of the line. if it is
        # measure probability of ending the line
        fill_dict(A2, (t_1, t), '<end>') # Here dictionary is second_order, key is the tuple (t_1,t) anf value is a fake
                                                # token "<end>". This "<end>" will ensure that our line should end during data generation
                                                # if we sample the end token. This is required otherwise our line will go forever.
      if i == 1:
        # measure distribution of second word given only first word
        fill_dict(A1, t_1, t)
      else:
        t_2 = tokens[i-2] # This is always the case we are looking i.e when i is not zero and i is not 1
        fill_dict(A2, (t_2, t_1), t) # key is the previous two words and value is the current word.

# Normalize the initial distribution

In [7]:
# normalize the initial distribution only because we have count only for it
pi_total = sum(pi.values()) # get the total count
for t, count in pi.items(): # t is a key, c is a value
    pi[t] = count / pi_total

# Create a function to convert list of possible words to dictionary of probabilities

In [8]:
def list_to_probdict(tokens): # This function does two things. First it creates the dictionary of count and then normalize
                    # the counts to convert each count into probability. The input to the function is ts which is
                    # list of the token
  d = {}
  n = len(tokens)  # The total number of samples which is length of ts
  for t in tokens: # loop through each token
    d[t] = d.get(t, 0.) + 1 # increment the value by 1 each time we encounter the token. After this for loop we have the dictionary
                            # of counts where key is the token and value is the corressponding count
  for t, c in d.items():
    d[t] = c / n
  return d

# Apply a function to first and second order Transitions

In [9]:
# Now we have a first order dictionary which stores second word of each sentence. we have previously stored in this
# dictionery is the list of possible next tokens. we replace the list of tokens by dictionary of probabilities
# by applting the function list2pdict
for t_1, token in A1.items():
  # replace list with dictionary of probabilities
  A1[t_1] = list_to_probdict(token)

In [10]:
for (t_2, t_1), token in A2.items():
  A2[(t_2, t_1)] = list_to_probdict(token)

# Write a function to sample the words

In [11]:
def sample_word(d): #Here input "d" is the dictionary of probability. key is the possible word and value is the corresponding probability
  p0 = np.random.random() # draw a sample from uniform distribution
  cumulative = 0
  for t, p in d.items():
    cumulative += p
    if p0 < cumulative:
      return t

# Function to generate the poem

In [12]:
def generate_Poem():
  for i in range(7): # generate 7 lines
    sentence = []

    # initial word
    w0 = sample_word(pi)
    sentence.append(w0)

    # sample second word
    w1 = sample_word(A1[w0])
    sentence.append(w1)

    # second-order transitions until END
    while True:
      w2 = sample_word(A2[(w0, w1)])
      if w2 == '<end>':
        break
      sentence.append(w2)
      w0 = w1 # update the previous word
      w1 = w2 # update the previous word
    print(' '.join(sentence))

In [16]:
generate_Poem()

may still come chasing in.
my heart with mud. an old wind
my heart with mud. an old wind
living world, hold me
resurrection fire. and me here
give way.
who will arrive in time to vanquish
